# Experiment 04 — Over-the-air adversarial perturbations

**Objective.** Generate a speaker-played, speech-independent repeating perturbation and measure whether it increases word error rate (WER) when a person speaks near a microphone. The attack maximizes the CTC loss of `facebook/wav2vec2-base-960h` on a *training* corpus while sampling approximate playback paths, level changes, and timing offsets. This is a research prototype: simulated performance does not establish physical effectiveness, and a notable WER increase is an empirical outcome, not a guarantee.

The speaker emits **only the perturbation**, not a modified recording of the speaker's voice. The signal is universal across utterances, although it is optimized for the training corpus and surrogate ASR. Evaluate on held-out speakers, sentences, rooms and, if possible, additional ASR systems. Conduct recordings with participant consent and keep levels comfortable; use a calibrated sound-level meter and your institution's hearing-safety and research protocols. Digital amplitude is not sound pressure level.

## 1. Setup and LibriSpeech data

Install dependencies using Experiment 1. Download `train-clean-100.tar.gz` (6.3 GB) and `test-clean.tar.gz` (346 MB) from [OpenSLR 12](https://www.openslr.org/12), verify their checksums against the linked `md5sum.txt`, and extract both archives into `../data/exp4/`. The expected paths are `../data/exp4/LibriSpeech/train-clean-100/` and `../data/exp4/LibriSpeech/test-clean/`. Allow additional disk space for extraction. Downloads are manual; this notebook does not fetch the archives. LibriSpeech is distributed under CC BY 4.0; cite the corpus and its creators when reporting results.

The loader reads the supplied FLAC audio and chapter transcript files directly. It samples 100 training and 20 test utterances reproducibly by default, saves their IDs, and checks speaker separation. Increase these limits for a substantial study. The test subset is for final evaluation: use a separate development split for tuning. `facebook/wav2vec2-base-960h` was trained on LibriSpeech training data, so attack-training clips overlap the surrogate's pretraining corpus; test-clean remains held out from attack optimization but is not an independent corpus for a broad transferability claim. Physical trials still require a separate recording manifest described below.

In [4]:
from pathlib import Path
import copy, json, random, re
import numpy as np
import pandas as pd
import librosa, soundfile as sf
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from jiwer import process_words
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
from IPython.display import Audio, display

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SR = 16000
ROOT = Path('../data/exp4')
OUT = Path('../results/exp4_ota'); OUT.mkdir(parents=True, exist_ok=True)
MODEL_NAME = 'facebook/wav2vec2-base-960h'
print('Device:', DEVICE, '| Output:', OUT.resolve())

C:\Users\Austin\Documents\Code\Adversarial Audio\adversarial-audio\gan.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda | Output: C:\Users\Austin\Documents\Code\Adversarial Audio\adversarial-audio\results\exp4_ota


In [5]:
def normalize_text(text):
    return re.sub(r'\s+', ' ', re.sub(r"[^A-Z' ]", ' ', str(text).upper())).strip()

LIBRISPEECH = ROOT / 'LibriSpeech'
TRAIN_LIMIT, TEST_LIMIT = 100, 20  # Set to None to use all utterances (large memory/runtime cost).

def load_librispeech(split, limit, seed):
    split_dir = LIBRISPEECH / split
    transcripts = sorted(split_dir.rglob('*.trans.txt')) if split_dir.is_dir() else []
    if not transcripts:
        raise FileNotFoundError(f'No transcript files in {split_dir}. Download and extract {split}.tar.gz as described above.')
    entries = []
    for transcript in transcripts:
        for line in transcript.read_text(encoding='utf-8').splitlines():
            parts = line.split(maxsplit=1)
            if len(parts) != 2:
                raise ValueError(f'Malformed transcript line in {transcript}: {line!r}')
            utterance_id, text = parts
            audio_path = transcript.parent / f'{utterance_id}.flac'
            if not audio_path.is_file(): raise FileNotFoundError(audio_path)
            reference = normalize_text(text)
            if not reference: raise ValueError(f'Empty transcript: {utterance_id}')
            entries.append((utterance_id, audio_path, reference))
    if len({entry[0] for entry in entries}) != len(entries): raise ValueError(f'Duplicate utterance IDs in {split}')
    if limit is not None:
        if limit <= 0: raise ValueError('Sample limit must be positive or None')
        entries = random.Random(seed).sample(entries, min(limit, len(entries)))
    items = []
    for utterance_id, audio_path, reference in entries:
        audio, _ = librosa.load(audio_path, sr=SR, mono=True)
        if len(audio) < SR // 2 or not np.isfinite(audio).all():
            raise ValueError(f'Invalid or too-short recording: {audio_path}')
        items.append({'id': utterance_id, 'speaker': utterance_id.split('-')[0],
                      'path': str(audio_path), 'audio': audio.astype(np.float32), 'reference': reference})
    return items

train = load_librispeech('train-clean-100', TRAIN_LIMIT, SEED)
heldout = load_librispeech('test-clean', TEST_LIMIT, SEED)
if {x['speaker'] for x in train} & {x['speaker'] for x in heldout}:
    raise ValueError('Training and test speakers overlap')
selection = [{'split': split, 'id': x['id'], 'speaker': x['speaker'], 'path': x['path'], 'reference': x['reference']}
             for split, items in [('train-clean-100', train), ('test-clean', heldout)] for x in items]
pd.DataFrame(selection).to_csv(OUT / 'librispeech_selection.csv', index=False)
print(f'Training clips: {len(train)}; held-out test clips: {len(heldout)}')

FileNotFoundError: Create ..\data\ota\train.csv with audio_path,reference columns (see instructions above).

## 2. Surrogate model and differentiable channel

The perturbation is a periodic waveform, optimized with projected gradient ascent on reference CTC loss. Each step samples a training utterance, circular phase, perturbation level, and a synthetic causal reverberant impulse response. A frequency mask constrains the **digital source** to a configurable band. These simplified paths do not model device nonlinearity, room geometry, independent speech-path reverberation, or automatic gain control; measured impulse responses and real recordings are needed to assess the gap. Changing playback volume changes received level and may change performance.

In [ ]:
processor = Wav2Vec2Processor.from_pretrained(MODEL_NAME)
model = Wav2Vec2ForCTC.from_pretrained(MODEL_NAME).to(DEVICE).eval()
for p in model.parameters(): p.requires_grad_(False)

def normalized(x):
    # Match Wav2Vec2 feature extractor's per-utterance zero-mean/unit-variance preprocessing.
    return (x - x.mean(dim=-1, keepdim=True)) / (x.var(dim=-1, keepdim=True, unbiased=False) + 1e-7).sqrt()

def transcribe(audio):
    x = torch.as_tensor(audio, dtype=torch.float32, device=DEVICE).reshape(1, -1)
    with torch.no_grad(): ids = model(normalized(x)).logits.argmax(dim=-1)
    return normalize_text(processor.batch_decode(ids)[0])

def counts(ref, hyp):
    result = process_words(ref, hyp)
    return {'substitutions': result.substitutions, 'deletions': result.deletions,
            'insertions': result.insertions, 'words': result.hits + result.substitutions + result.deletions, 'wer': result.wer}

In [ ]:
# These are digital amplitudes, not acoustic SPL. Tune only on training/development data.
config = dict(duration_s=2.0, epsilon=0.04, step=0.001, iterations=200,
              low_hz=100, high_hz=7000, gain_min=0.3, gain_max=1.0, rir_ms=80)
length = int(config['duration_s'] * SR)

def bandlimit(x):
    freqs = torch.fft.rfftfreq(x.numel(), d=1/SR, device=x.device)
    mask = (freqs >= config['low_hz']) & (freqs <= config['high_hz'])
    return torch.fft.irfft(torch.fft.rfft(x) * mask, n=x.numel())

def project(x):
    # Scaling after the Fourier projection preserves both band and peak constraints.
    x = bandlimit(x)
    return x * torch.clamp(config['epsilon'] / x.abs().max().clamp_min(1e-12), max=1.0)

def room_response():
    size = max(2, int(SR * config['rir_ms'] / 1000))
    t = torch.arange(size, device=DEVICE) / SR
    tail = torch.randn(size, device=DEVICE) * torch.exp(-t / random.uniform(0.01, 0.04)) * 0.02
    tail[0] += 1.0
    return tail / tail.abs().sum()

def received(delta, samples, phase, gain, rir):
    index = (torch.arange(samples, device=DEVICE) + phase) % delta.numel()
    signal = delta[index][None, None, :]
    # conv1d is correlation, so reverse the causal impulse response.
    return gain * F.conv1d(F.pad(signal, (rir.numel()-1, 0)), rir.flip(0)[None, None, :]).flatten()

delta = torch.zeros(length, device=DEVICE)
history = []
for iteration in range(config['iterations']):
    item = random.choice(train)
    speech = torch.as_tensor(item['audio'], device=DEVICE)
    labels = processor.tokenizer(item['reference'], return_tensors='pt').input_ids.to(DEVICE)
    delta = delta.detach().requires_grad_(True)
    phase = random.randrange(length)
    gain = random.uniform(config['gain_min'], config['gain_max'])
    mixture = (speech + received(delta, speech.numel(), phase, gain, room_response())).clamp(-1, 1)
    loss = model(normalized(mixture[None, :]), labels=labels).loss
    if not torch.isfinite(loss): raise RuntimeError('Non-finite CTC loss; check audio and transcript lengths.')
    grad = torch.autograd.grad(loss, delta)[0]
    with torch.no_grad(): delta = project(delta + config['step'] * grad.sign())
    history.append(float(loss.detach()))
    if (iteration + 1) % 20 == 0: print(f'{iteration+1}: CTC loss {history[-1]:.2f}')

perturbation = delta.detach().cpu().numpy()
print('Peak:', np.max(np.abs(perturbation)), 'RMS:', np.sqrt(np.mean(perturbation**2)))
plt.plot(history); plt.xlabel('Iteration'); plt.ylabel('Sampled CTC loss'); plt.show()

## 3. Export and simulated held-out check

The control is band-limited random noise matched to the perturbation's digital RMS. Adjust it downward if matching RMS would exceed the configured peak limit. Do not normalize audio in the player: that would change the intended level. In the real experiment, adjust the two playback levels to the same **measured received level** at the microphone, not merely the same digital RMS. The periodic boundary may be audible; inspect and listen at a low level.

In [ ]:
noise = project(torch.randn(length, device=DEVICE)).detach().cpu().numpy()
target_rms = np.sqrt(np.mean(perturbation**2))
noise *= min(target_rms / max(np.sqrt(np.mean(noise**2)), 1e-12), config['epsilon'] / max(np.max(np.abs(noise)), 1e-12))
sf.write(OUT / 'perturbation.wav', perturbation, SR, subtype='FLOAT')
sf.write(OUT / 'noise_control.wav', noise, SR, subtype='FLOAT')
(OUT / 'config.json').write_text(json.dumps({'seed': SEED, 'model': MODEL_NAME, 'sample_rate': SR, 'train_split': 'train-clean-100', 'test_split': 'test-clean', 'train_limit': TRAIN_LIMIT, 'test_limit': TEST_LIMIT, **config}, indent=2))
print('Noise RMS:', np.sqrt(np.mean(noise**2)), 'Attack RMS:', target_rms)
display(Audio(perturbation, rate=SR, normalize=False))

In [ ]:
# Fixed conditions make this a reproducible diagnostic, not a physical result.
rows = []
rir = torch.zeros(1, device=DEVICE); rir[0] = 1
for item in heldout:
    speech = item['audio']
    for condition, signal in [('clean', None), ('noise', noise), ('attack', perturbation)]:
        mixed = speech.copy()
        if signal is not None:
            source = torch.as_tensor(signal, device=DEVICE)
            mixed = np.clip(mixed + received(source, len(mixed), 0, 0.6, rir).cpu().numpy(), -1, 1)
        hyp = transcribe(mixed)
        rows.append({'path': item['path'], 'condition': condition, 'reference': item['reference'],
                     'hypothesis': hyp, **counts(item['reference'], hyp)})
simulated = pd.DataFrame(rows)
display(simulated[['path', 'condition', 'reference', 'hypothesis', 'wer']])

## Cross-domain evaluation: Med-Dictate

[Med-Dictate](https://huggingface.co/datasets/corti/med-dictate) contains English medical dictations (24 test recordings) alongside other languages. Use its English `en` configuration and `test` split with the existing English ASR. The dataset card calls for citation of Nix et al., [*Symphony for Speech-to-Text*](https://doi.org/10.48550/arXiv.2605.16545), and states that the audio is licensed for evaluation, not model or attack development. Review the linked license and do not use these examples to optimize or tune the perturbation, select hyperparameters, or train a model. Freeze the attack and evaluation plan before running this section; use a separate development set for tuning. Downloading the dataset requires network access and the `datasets` package installed through Experiment 1.

Use the verbatim `transcription` as the reference, including spoken formatting commands; `transcription_formatted` is not a verbatim ASR target. Recordings are long, so inference uses consecutive 10-second chunks and concatenates hypotheses before scoring the whole recording. Words split at chunk boundaries can increase baseline errors; all conditions use identical boundaries. This is a **simulated digital mixture**, not an over-the-air result. Compare corpus WER against both clean and noise controls; the small number of recordings and any differences in speaker, content, and recording conditions limit claims about domain generalization.

In [ ]:
from datasets import Audio as DatasetAudio, load_dataset
from io import BytesIO

MED_CONFIG = 'en'
MED_SPLIT = 'test'
MED_CHUNK_SECONDS = 10
MED_GAIN = 0.6  # Match the fixed gain used for the LibriSpeech diagnostic.
med = load_dataset('corti/med-dictate', MED_CONFIG, split=MED_SPLIT)
# Decode locally with soundfile rather than requiring an additional audio decoder.
med = med.cast_column('audio', DatasetAudio(decode=False))
print(f'Med-Dictate: {len(med)} recordings; dataset fingerprint: {med._fingerprint}')

def med_audio(record):
    encoded = record['audio']
    if encoded.get('bytes') is not None:
        source = BytesIO(encoded['bytes'])
    elif encoded.get('path'):
        source = encoded['path']
    else:
        raise ValueError(f"Missing audio for {record['id']}")
    audio, rate = sf.read(source, dtype='float32', always_2d=True)
    audio = audio.mean(axis=1)
    if rate != SR: audio = librosa.resample(audio, orig_sr=rate, target_sr=SR)
    if not len(audio) or not np.isfinite(audio).all(): raise ValueError(f"Invalid audio for {record['id']}")
    return audio

def med_decode(audio, signal):
    chunk_size = MED_CHUNK_SECONDS * SR
    source = None if signal is None else torch.as_tensor(signal, device=DEVICE)
    parts = []
    rir = torch.ones(1, device=DEVICE)
    for start in range(0, len(audio), chunk_size):
        segment = audio[start:start + chunk_size].copy()
        if source is not None:
            # Keep the source phase continuous across inference chunks.
            segment = np.clip(segment + received(source, len(segment), start % len(signal), MED_GAIN, rir).cpu().numpy(), -1, 1)
        # A very short final tail may be too short for the model convolution stack.
        if len(segment) < SR // 2: segment = np.pad(segment, (0, SR // 2 - len(segment)))
        parts.append(transcribe(segment))
    return normalize_text(' '.join(parts))

med_rows = []
for record in med:
    audio = med_audio(record)
    reference = normalize_text(record['transcription'])
    if not reference: raise ValueError(f"Empty reference for {record['id']}")
    for condition, signal in [('clean', None), ('noise', noise), ('attack', perturbation)]:
        hypothesis = med_decode(audio, signal)
        med_rows.append({'id': record['id'], 'condition': condition, 'reference': reference,
                         'hypothesis': hypothesis, **counts(reference, hypothesis)})
    print(f"Completed {record['id']}")
med_results = pd.DataFrame(med_rows)
med_results.to_csv(OUT / 'med_dictate_results.csv', index=False)
med_totals = med_results.groupby('condition')[['substitutions', 'deletions', 'insertions', 'words']].sum()
med_totals['wer'] = (med_totals.substitutions + med_totals.deletions + med_totals.insertions) / med_totals.words
display(med_totals)
print(f"Attack minus clean: {(med_totals.loc['attack', 'wer'] - med_totals.loc['clean', 'wer'])*100:.2f} percentage points")
print(f"Attack minus noise: {(med_totals.loc['attack', 'wer'] - med_totals.loc['noise', 'wer'])*100:.2f} percentage points")
display(med_results[['id', 'condition', 'reference', 'hypothesis', 'wer']])

## 4. Physical protocol

1. Fix the speaker, microphone, room, distances, gain settings, and recording format. Use the same target ASR and preprocessing for every condition; document devices, placement, sound level, and model version. Avoid clipping and record any automatic gain or noise suppression.
2. For each held-out sentence, collect separate microphone recordings under **silence**, **noise control**, and **attack**. Play the corresponding WAV continuously in a loop before and during speech for the latter two conditions. Use a player that preserves amplitude and does not insert gaps; log actual playback and recording settings. Randomize condition order, repeat trials, and keep speakers blind to condition where practical. Random timing between speech onset and playback is desirable. Do not digitally add the source WAV to these recordings.
3. Capture the same intended sentence in each condition, and independently transcribe what was actually spoken. Exclude or separately flag misread trials using predefined criteria. Keep the reference independent of ASR output.
4. Create `../data/exp4/physical.csv` with columns `trial_id,pair_id,condition,audio_path,reference`. Use `silence`, `noise`, or `attack`; `pair_id` identifies the same speaker/sentence/repetition across the three conditions, with exactly one recording per condition. Paths are relative to the CSV. Preserve any metadata (speaker, room, position, measured SPL, onset timing) in additional columns and record deviations.
5. Freeze the attack and evaluation plan before collecting held-out physical data. Compare attack–silence and attack–noise in percentage points. Random noise controls for ordinary masking; a higher attack WER alone does not prove an adversarial effect if the noise control performs similarly.

In [ ]:
physical_path = ROOT / 'physical.csv'
if not physical_path.exists():
    print(f'Physical evaluation pending: create {physical_path} after recording.')
else:
    trials = pd.read_csv(physical_path, keep_default_na=False)
    required = {'trial_id', 'pair_id', 'condition', 'audio_path', 'reference'}
    if not required.issubset(trials.columns) or trials.empty: raise ValueError('Invalid physical manifest')
    if trials.trial_id.duplicated().any(): raise ValueError('trial_id must be unique')
    if set(trials.condition) != {'silence', 'noise', 'attack'}: raise ValueError('Expected silence, noise, and attack conditions')
    group_counts = trials.groupby(['pair_id', 'condition']).size().unstack(fill_value=0)
    if not (group_counts == 1).all().all(): raise ValueError('Each pair must have exactly one trial per condition')
    trials['reference'] = trials.reference.map(normalize_text)
    if (trials.reference == '').any(): raise ValueError('References must not be empty')
    if (trials.groupby('pair_id').reference.nunique() != 1).any(): raise ValueError('References differ within a pair; inspect misreads')
    records = []
    for _, row in trials.iterrows():
        audio, _ = librosa.load(physical_path.parent / row.audio_path, sr=SR, mono=True)
        if len(audio) < SR // 2 or not np.isfinite(audio).all(): raise ValueError(f"Invalid audio: {row.audio_path}")
        hyp = transcribe(audio)
        records.append({**row.to_dict(), 'hypothesis': hyp, **counts(row.reference, hyp)})
    results = pd.DataFrame(records)
    results.to_csv(OUT / 'physical_results.csv', index=False)
    totals = results.groupby('condition')[['substitutions', 'deletions', 'insertions', 'words']].sum()
    totals['wer'] = (totals.substitutions + totals.deletions + totals.insertions) / totals.words
    display(totals)
    paired = results.pivot(index='pair_id', columns='condition', values='wer')
    print(f"Attack minus silence: {(totals.loc['attack', 'wer'] - totals.loc['silence', 'wer'])*100:.2f} percentage points")
    print(f"Attack minus noise: {(totals.loc['attack', 'wer'] - totals.loc['noise', 'wer'])*100:.2f} percentage points")
    display(results[['trial_id', 'pair_id', 'condition', 'reference', 'hypothesis', 'wer']])

## 5. Interpretation and reporting

WER = (substitutions + deletions + insertions) / reference words; it can exceed 100%. The table uses corpus-level WER (summed errors divided by summed reference words), and differences are absolute percentage points. Report trial counts, individual transcripts, failures, condition levels, and uncertainty across independent speakers or sessions; repeated trials of one speaker are not independent subjects. Predefine a practically meaningful increase and a statistical analysis, including confidence intervals, before interpreting results. A null or negative difference is a valid result. Record separately any clipping, recording failures, sentence errors, hardware changes, and deviations. This notebook does not establish inaudibility or physical success without the physical measurements.